# Tutorial: TL-007 Local Model Clients Explorer

Audience:
- Engineers working on `personal_kb` model boundaries.

Prerequisites:
- The repository dependencies are installed with `uv sync`.
- You know the difference between an LLM client, embedding model, reranker, and extraction adapter.
- Optional: LM Studio is running locally on `http://localhost:1234/v1`.

Learning goals:
- Inspect the TL-007 config-facing defaults.
- See how the LLM client surfaces visible text and `reasoning_content` separately.
- Verify structured JSON retry behavior and validator hooks.
- Demonstrate deterministic embedding normalization and reranker ordering with lightweight stubs.


## Outline

1. Locate the repo and import the TL-007 clients.
2. Inspect the current default config contract.
3. Demonstrate text generation plus reasoning metadata with a stubbed OpenAI-compatible chat client.
4. Demonstrate structured JSON retry and extraction validator retries.
5. Demonstrate embedding normalization and deterministic reranking with local stubs.
6. Optionally run a tiny live LM Studio smoke check.
7. Review pitfalls, then try one exercise.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
import json
import os
from pathlib import Path
from typing import Any

from pydantic import BaseModel

from personal_kb.core.config_loader import load_config
from personal_kb.models.embedding_client import EmbeddingClient
from personal_kb.models.llm_client import LLMClient, LLMResponseMetadata, LLMTextResponse, StructuredLLMResult
from personal_kb.models.reranker_client import RerankerClient
from personal_kb.models.structured_extraction_client import StructuredExtractionClient
from personal_kb.schemas.common import ScoreBreakdown
from personal_kb.schemas.config import EmbeddingConfig, LLMConfig, RerankerConfig
from personal_kb.schemas.search import SearchDocumentResult


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "personal_kb").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
RUN_LIVE_SMOKE = os.environ.get("TL007_RUN_LIVE_SMOKE", "0") == "1"

print({"repo_root": REPO_ROOT.as_posix(), "run_live_smoke": RUN_LIVE_SMOKE})


{'repo_root': '/Users/pelmeshek1706/Desktop/projects/knowledge_agent', 'run_live_smoke': False}


## Step 1 - Inspect the config contract

TL-007 changed the config-facing defaults in one important way: the public surface now defaults to `non_thinking` mode while preserving the user-approved temporary LM Studio model id for this repo.


In [ ]:
import os
{k: v for k, v in os.environ.items() if k.startswith(("LMSTUDIO", "LM_STUDIO", "PERSONAL_KB_"))}


{'LMSTUDIO_URL': 'http://localhost:1234',
 'LMSTUDIO_EMBED_FALLBACK': 'true',
 'LMSTUDIO_MODEL': 'qwen2.5-1.5b-instruct',
 'LMSTUDIO_EMBED_MODEL': 'text-embedding-nomic-embed-text-v1.5'}

In [6]:
os.environ.pop("LMSTUDIO_MODEL", None)
os.environ.pop("LMSTUDIO_URL", None)
os.environ.pop("LMSTUDIO_EMBED_MODEL", None)


'text-embedding-nomic-embed-text-v1.5'

In [7]:
config = load_config(REPO_ROOT / "configs" / "default.yaml")
config.models.llm.model_name


'mlx-qwen3.5-9b-claude-4.6-opus-reasoning-distilled-v2'

In [8]:
config = load_config(REPO_ROOT / "configs" / "default.yaml")

config_summary = {
    "llm": {
        "provider": config.models.llm.provider,
        "base_url": config.models.llm.base_url,
        "model_name": config.models.llm.model_name,
        "default_thinking_mode": config.models.llm.default_thinking_mode,
        "structured_output_retries": config.models.llm.structured_output_retries,
    },
    "embedding": {
        "model_name": config.models.embedding.model_name,
        "dimension": config.models.embedding.dimension,
        "normalize_embeddings": config.models.embedding.normalize_embeddings,
    },
    "reranker": {
        "model_name": config.models.reranker.model_name,
        "top_k_after_rerank": config.models.reranker.top_k_after_rerank,
    },
}

config_summary


{'llm': {'provider': 'lmstudio_openai_compatible',
  'base_url': 'http://localhost:1234/v1',
  'model_name': 'mlx-qwen3.5-9b-claude-4.6-opus-reasoning-distilled-v2',
  'default_thinking_mode': 'non_thinking',
  'structured_output_retries': 2},
 'embedding': {'model_name': 'Qwen/Qwen3-Embedding-0.6B',
  'dimension': 2048,
  'normalize_embeddings': True},
 'reranker': {'model_name': 'Qwen/Qwen3-Reranker-0.6B',
  'top_k_after_rerank': 8}}

## Step 2 - Stub the LLM client boundary

The next cell uses a tiny fake OpenAI-compatible client so you can inspect TL-007 behavior without depending on a live model. The important parts are:
- request-side thinking-mode control;
- separate `content` and `reasoning_content` fields;
- metadata that makes provider caveats visible instead of silent.


In [9]:
@dataclass
class FakeCompletion:
    payload: dict[str, Any]

    def model_dump(self, mode: str = "json") -> dict[str, Any]:
        return self.payload


class FakeCompletions:
    def __init__(self, payloads: list[dict[str, Any]]) -> None:
        self.payloads = [FakeCompletion(payload) for payload in payloads]
        self.requests: list[dict[str, Any]] = []

    def create(self, **kwargs: Any) -> FakeCompletion:
        self.requests.append(kwargs)
        return self.payloads.pop(0)


class FakeChatAPI:
    def __init__(self, payloads: list[dict[str, Any]]) -> None:
        self.completions = FakeCompletions(payloads)


class FakeClient:
    def __init__(self, payloads: list[dict[str, Any]]) -> None:
        self.chat = FakeChatAPI(payloads)


def response_payload(content: str, reasoning: str | None = None) -> dict[str, Any]:
    return {
        "choices": [
            {
                "finish_reason": "stop",
                "message": {
                    "content": content,
                    "reasoning_content": reasoning,
                },
            }
        ],
        "usage": {
            "prompt_tokens": 10,
            "completion_tokens": 7,
            "total_tokens": 17,
            "completion_tokens_details": {"reasoning_tokens": 5},
        },
    }


fake_client = FakeClient([
    response_payload(
        content="Visible answer from the assistant.",
        reasoning="Reasoning text that the current runtime might still expose.",
    )
])
llm_client = LLMClient(LLMConfig(), client_factory=lambda _: fake_client)
text_response = llm_client.generate_text("Summarize TL-007 in one sentence.")

{
    "request_extra_body": fake_client.chat.completions.requests[0].get("extra_body"),
    "content": text_response.content,
    "reasoning_content": text_response.reasoning_content,
    "metadata": text_response.metadata.model_dump(mode="json"),
}


{'request_extra_body': {'chat_template_kwargs': {'enable_thinking': False}},
 'content': 'Visible answer from the assistant.',
 'reasoning_content': 'Reasoning text that the current runtime might still expose.',
 'metadata': {'provider': 'lmstudio_openai_compatible',
  'model_name': 'mlx-qwen3.5-9b-claude-4.6-opus-reasoning-distilled-v2',
  'finish_reason': 'stop',
  'thinking_mode_requested': 'non_thinking',
  'thinking_mode_defaulted': True,
  'reasoning_content_present': True,
  'provider_honored_non_thinking_request': False,
  'retry_count': 0,
  'usage': {'prompt_tokens': 10,
   'completion_tokens': 7,
   'total_tokens': 17,
   'reasoning_tokens': 5},
  'warnings': ['Provider returned reasoning content for a non-thinking request.'],
  'raw_provider_response': {'choices': [{'finish_reason': 'stop',
     'message': {'content': 'Visible answer from the assistant.',
      'reasoning_content': 'Reasoning text that the current runtime might still expose.'}}],
   'usage': {'prompt_tokens

## Step 3 - Structured JSON retry and extraction validators

The LLM client can ask LM Studio for JSON Schema output, but TL-007 still validates locally and retries when the payload is invalid. The extraction client adds a second validation hook for task-specific business rules without drifting into TL-008 orchestration.


In [10]:
class ExtractionPayload(BaseModel):
    label: str


retry_client = FakeClient([
    response_payload('{"label": 123}'),
    response_payload('{"label": "valid-json"}'),
])
json_llm_client = LLMClient(
    LLMConfig(structured_output_retries=1),
    client_factory=lambda _: retry_client,
)
json_result = json_llm_client.generate_json(
    "Return a JSON object with a string label.",
    response_schema=ExtractionPayload,
)


@dataclass
class FakeStructuredLLM:
    results: list[StructuredLLMResult[ExtractionPayload]]
    config: LLMConfig = field(default_factory=lambda: LLMConfig(structured_output_retries=1))

    def __post_init__(self) -> None:
        self.calls: list[dict[str, Any]] = []

    def generate_json(self, prompt: str, **kwargs: Any) -> StructuredLLMResult[ExtractionPayload]:
        self.calls.append({"prompt": prompt, **kwargs})
        return self.results.pop(0)


def structured_result(label: str) -> StructuredLLMResult[ExtractionPayload]:
    return StructuredLLMResult(
        value=ExtractionPayload(label=label),
        response=LLMTextResponse(
            content=json.dumps({"label": label}),
            metadata=LLMResponseMetadata(
                provider="lmstudio_openai_compatible",
                model_name="stub-model",
                thinking_mode_requested="non_thinking",
            ),
        ),
        attempts=1,
    )


structured_llm = FakeStructuredLLM([
    structured_result("reject-me"),
    structured_result("accept-me"),
])
structured_client = StructuredExtractionClient(structured_llm)


def validator(payload: ExtractionPayload) -> ExtractionPayload:
    if payload.label != "accept-me":
        raise ValueError("label must be accept-me")
    return payload


structured_result_value = structured_client.extract(
    "Extract a label.",
    response_schema=ExtractionPayload,
    validator=validator,
    max_retries=1,
)

{
    "json_retry_attempts": json_result.attempts,
    "json_value": json_result.value.model_dump(mode="json"),
    "validator_attempts": structured_result_value.attempts,
    "validator_notes": structured_result_value.validator_notes,
    "repaired_prompt_preview": structured_llm.calls[1]["prompt"],
}


{'json_retry_attempts': 2,
 'json_value': {'label': 'valid-json'},
 'validator_attempts': 2,
 'validator_notes': ['label must be accept-me'],
 'repaired_prompt_preview': 'Extract a label.\n\nPrevious validated JSON candidate: {"label":"reject-me"}\nAdditional validator feedback: label must be accept-me\nReturn a corrected JSON object that satisfies every requirement.'}

## Step 4 - Embedding normalization and deterministic reranking

The embedding and reranker examples stay lightweight and deterministic by using stubs. This is enough to demonstrate TL-007 boundary behavior without forcing model downloads inside the notebook.


In [11]:
class FakeEmbeddingBackend:
    def __init__(self, vectors: list[list[float]]) -> None:
        self.vectors = vectors
        self.calls: list[tuple[list[str], dict[str, Any]]] = []

    def encode(self, sentences: list[str], **kwargs: Any) -> list[list[float]]:
        self.calls.append((sentences, kwargs))
        return self.vectors


class FakeRerankerBackend:
    def __init__(self, scores: list[float]) -> None:
        self.scores = scores
        self.calls: list[tuple[list[tuple[str, str]], dict[str, Any]]] = []

    def predict(self, sentences: list[tuple[str, str]], **kwargs: Any) -> list[float]:
        self.calls.append((sentences, kwargs))
        return self.scores


embedding_backend = FakeEmbeddingBackend([
    [3.0, 4.0] + [0.0] * 1022,
    [5.0, 12.0] + [0.0] * 1022,
])
embedding_client = EmbeddingClient(
    EmbeddingConfig(),
    backend_factory=lambda _: embedding_backend,
)
vectors = embedding_client.embed_batch(["alpha", "beta"], instruction="Index this text")


def candidate(document_id: str, summary: str) -> SearchDocumentResult:
    return SearchDocumentResult(
        document_id=document_id,
        title=document_id,
        file_path=f"data/{document_id}.md",
        document_type="markdown",
        summary=summary,
        confidence=0.5,
        score_breakdown=ScoreBreakdown(final_score=0.5),
    )


reranker_backend = FakeRerankerBackend([0.2, 0.9, 0.9])
reranker_client = RerankerClient(
    RerankerConfig(),
    backend_factory=lambda _: reranker_backend,
)
ranked = reranker_client.rerank(
    "query",
    [candidate("doc-a", "alpha"), candidate("doc-b", "beta"), candidate("doc-c", "gamma")],
    top_k=3,
)

{
    "embedding_dimension": len(vectors[0]),
    "embedding_norms": [round(sum(value * value for value in vector), 6) for vector in vectors],
    "embedding_request_preview": embedding_backend.calls[0][0][0],
    "reranked_document_ids": [item.document_id for item in ranked],
    "reranker_scores": [item.score_breakdown.reranker_score for item in ranked],
}


{'embedding_dimension': 1024,
 'embedding_norms': [1.0, 1.0],
 'embedding_request_preview': 'Instruction: Index this text\nText: alpha',
 'reranked_document_ids': ['doc-b', 'doc-c', 'doc-a'],
 'reranker_scores': [0.9, 0.9, 0.2]}

## Step 5 - Optional live LM Studio smoke check

Set `TL007_RUN_LIVE_SMOKE=1` before launching Jupyter if you want to hit the running LM Studio server. Keep this smoke test tiny because the current live model may ignore the non-thinking toggle and may spend too many tokens in `reasoning_content` if the budget is too small.


In [12]:
if not RUN_LIVE_SMOKE:
    live_summary = {"skipped": True, "reason": "Set TL007_RUN_LIVE_SMOKE=1 to enable the live call."}
else:
    live_client = LLMClient(config.models.llm)
    live_response = live_client.generate_text(
        "Reply with the single word READY.",
        max_tokens=64,
    )
    live_summary = {
        "skipped": False,
        "content": live_response.content,
        "reasoning_content": live_response.reasoning_content,
        "metadata": live_response.metadata.model_dump(mode="json"),
    }

live_summary


{'skipped': True,
 'reason': 'Set TL007_RUN_LIVE_SMOKE=1 to enable the live call.'}

## Pitfalls and extensions

- Common mistake: assuming that a `non_thinking` request guarantees no reasoning output. The current live runtime can still return `reasoning_content`, so callers must inspect metadata instead of assuming the toggle was honored.
- Common mistake: using too small a token budget for reasoning-capable models. You can get an empty visible answer even when reasoning tokens were consumed.
- Extension: replace the stubbed embedding and reranker backends with live local models on a machine where those downloads are available.

## Exercises

1. Re-run the text-generation example with `thinking_mode="thinking"` and compare the request payload.
2. Set the live smoke flag and inspect whether the current LM Studio runtime still returns reasoning content for a non-thinking request.
3. Change the embedding vectors so one is already unit-normalized and confirm the contract still preserves batch order.


In [13]:
# Exercise answer scaffold
thinking_demo_client = LLMClient(
    LLMConfig(),
    client_factory=lambda _: FakeClient([response_payload("READY")]),
)
thinking_demo_response = thinking_demo_client.generate_text(
    "Reply with READY.",
    thinking_mode="thinking",
)

{
    "thinking_request_extra_body": thinking_demo_client._get_client().chat.completions.requests[0]["extra_body"],
    "thinking_response": thinking_demo_response.model_dump(mode="json"),
}


{'thinking_request_extra_body': {'chat_template_kwargs': {'enable_thinking': True}},
 'thinking_response': {'content': 'READY',
  'reasoning_content': None,
  'metadata': {'provider': 'lmstudio_openai_compatible',
   'model_name': 'mlx-qwen3.5-9b-claude-4.6-opus-reasoning-distilled-v2',
   'finish_reason': 'stop',
   'thinking_mode_requested': 'thinking',
   'thinking_mode_defaulted': False,
   'reasoning_content_present': False,
   'provider_honored_non_thinking_request': None,
   'retry_count': 0,
   'usage': {'prompt_tokens': 10,
    'completion_tokens': 7,
    'total_tokens': 17,
    'reasoning_tokens': 5},
   'warnings': [],
   'raw_provider_response': {'choices': [{'finish_reason': 'stop',
      'message': {'content': 'READY', 'reasoning_content': None}}],
    'usage': {'prompt_tokens': 10,
     'completion_tokens': 7,
     'total_tokens': 17,
     'completion_tokens_details': {'reasoning_tokens': 5}}}}}}